# Inference-Only: Test Set Prediction + Combine with Val
Loads saved checkpoints, predicts on `test/` only, merges with existing `submission.csv`.

**Inputs required in Kaggle:**
1. Competition dataset attached (contains `train/`, `val/`, `test/`)
2. Previous notebook output saved as a dataset (contains `.pth` checkpoints + `submission.csv`)

In [1]:
import os, gc, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

warnings.filterwarnings('ignore')

class CFG:
    device      = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 126
    num_workers = 2
    img_size    = 224
    max_time    = 48
    use_tta     = True

print(f'Device: {CFG.device}')


Device: cuda


In [4]:
from pathlib import Path

# ── Locate competition data ──────────────────────────────────────────────────
def find_data():
    candidates = [
        Path('/kaggle/input/competitions/cvpr-mslr-2026-track-2'),
        Path('/kaggle/input/cvpr-mslr-2026-track-2'),
    ]
    
    for p in candidates:
        if (p / 'test').exists():   # only check test (since that's what you showed)
            return p
    
    raise FileNotFoundError('Competition dataset not found!')

DATA_ROOT = find_data()
TEST_DIR  = DATA_ROOT / 'test'

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

# ── Find checkpoints + submission CSV ─────────────────────────────────────────
INPUT_ROOT = Path('/kaggle/input')

# IMPORTANT: your files are .pt (not .pth)
ckpt_files = sorted(INPUT_ROOT.rglob('*.pt'))

# submission file
val_csv_candidates = sorted(INPUT_ROOT.rglob('submission.csv'))

# ── Debug prints ──────────────────────────────────────────────────────────────
print(f'DATA ROOT : {DATA_ROOT}')
print(f'Test dir  : {TEST_DIR}')

print(f'\nCheckpoints found: {len(ckpt_files)}')
for c in ckpt_files:
    print(f'  {c}')

print(f'\nVal CSVs found: {len(val_csv_candidates)}')
for c in val_csv_candidates:
    print(f'  {c}')

DATA ROOT : /kaggle/input/competitions/cvpr-mslr-2026-track-2
Test dir  : /kaggle/input/competitions/cvpr-mslr-2026-track-2/test

Checkpoints found: 10
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_convnext_tiny_fb_in22k_ft_in1k_f0.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_convnext_tiny_fb_in22k_ft_in1k_f1.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_convnext_tiny_fb_in22k_ft_in1k_f2.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_convnext_tiny_fb_in22k_ft_in1k_f3.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_convnext_tiny_fb_in22k_ft_in1k_f4.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_tf_efficientnetv2_s_in21k_ft_in1k_f0.pt
  /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/best_tf_efficientnetv2_s_in21k_ft_in1k_f1.pt
  /kaggle/input/models/

In [7]:
class RTMDataset(Dataset):
    def __init__(self, sample_list, img_size=224, max_T=48):
        self.samples  = sample_list
        self.img_size = img_size
        self.max_T    = max_T

    def __len__(self):
        return len(self.samples)

    def _load(self, sample_dir):
        sd = Path(sample_dir)
        sid = sd.name
        rtms = []
        for i in range(1, 4):
            arr = np.load(str(sd / f'{sid}_RTM{i}.npy')).astype(np.float32)
            rtms.append(arr)
        stacked = np.stack(rtms, axis=0)   # (3, T, 256)
        return stacked.transpose(0, 2, 1)  # (3, 256, T)

    def _normalize(self, x):
        mn, mx = x.min(), x.max()
        return (x - mn) / (mx - mn + 1e-8)

    def _pad_time(self, rtm):
        C, H, T = rtm.shape
        if T >= self.max_T:
            s = (T - self.max_T) // 2
            return rtm[:, :, s:s + self.max_T]
        pad = self.max_T - T
        pl, pr = pad // 2, pad - pad // 2
        return np.pad(rtm, ((0,0),(0,0),(pl,pr)), mode='constant')

    def _resize(self, t):
        return F.interpolate(
            t.unsqueeze(0), size=(self.img_size, self.img_size),
            mode='bilinear', align_corners=False
        ).squeeze(0)

    def __getitem__(self, idx):
        sd, sid = self.samples[idx]
        rtm = self._load(sd)
        rtm = self._normalize(rtm)
        rtm = self._pad_time(rtm)
        img = self._resize(torch.from_numpy(rtm.copy()).float())
        return img, sid


def build_samples(split_dir):
    samples = []
    for d in sorted(Path(split_dir).iterdir()):
        if d.is_dir() and d.name.startswith('SAMPLE_'):
            if (d / f'{d.name}_RTM1.npy').exists():
                samples.append((str(d), d.name))
    return samples

test_samples = build_samples(TEST_DIR)
print(f'Test samples: {len(test_samples)}')


Test samples: 4914


In [8]:
@torch.no_grad()
def predict_tta(model, loader, use_tta=True):
    model.eval()
    probs_list, ids_list = [], []
    for imgs, sids in loader:
        imgs = imgs.to(CFG.device, non_blocking=True)
        with autocast():
            p = F.softmax(model(imgs), dim=1)
        if use_tta:
            with autocast():
                p_flip = F.softmax(model(torch.flip(imgs, [3])), dim=1)
            p = (p + p_flip) / 2.0
        probs_list.append(p.cpu())
        ids_list.extend(sids if isinstance(sids[0], str) else sids.tolist())
    return torch.cat(probs_list), ids_list


test_ds     = RTMDataset(test_samples, img_size=CFG.img_size, max_T=CFG.max_time)
test_loader = DataLoader(test_ds, batch_size=96, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=True)

ensemble_probs = None
sample_ids     = None

assert ckpt_files, 'No .pth checkpoints found! Did you add the previous notebook output as an input dataset?'

for i, path in enumerate(ckpt_files):
    ckpt  = torch.load(path, map_location='cpu', weights_only=False)
    mname = ckpt['name']
    acc   = ckpt.get('acc', 0)
    fold  = ckpt.get('fold', '?')
    print(f'[{i+1}/{len(ckpt_files)}] {mname}  fold={fold}  val_acc={acc:.1f}%')

    model = timm.create_model(mname, pretrained=False, num_classes=CFG.num_classes,
                              drop_rate=0, drop_path_rate=0)
    model.load_state_dict(ckpt['model'])
    model = model.to(CFG.device)

    probs, ids = predict_tta(model, test_loader, use_tta=CFG.use_tta)
    ensemble_probs = probs if ensemble_probs is None else ensemble_probs + probs
    if sample_ids is None:
        sample_ids = ids

    del model, ckpt; gc.collect(); torch.cuda.empty_cache()

ensemble_probs /= len(ckpt_files)
test_preds = ensemble_probs.argmax(dim=1).numpy()
print(f'Predictions: {len(test_preds)} samples, {len(np.unique(test_preds))} unique classes')


[1/10] convnext_tiny.fb_in22k_ft_in1k  fold=0  val_acc=83.0%
[2/10] convnext_tiny.fb_in22k_ft_in1k  fold=1  val_acc=83.4%
[3/10] convnext_tiny.fb_in22k_ft_in1k  fold=2  val_acc=84.1%
[4/10] convnext_tiny.fb_in22k_ft_in1k  fold=3  val_acc=79.4%
[5/10] convnext_tiny.fb_in22k_ft_in1k  fold=4  val_acc=83.8%
[6/10] tf_efficientnetv2_s.in21k_ft_in1k  fold=0  val_acc=82.1%
[7/10] tf_efficientnetv2_s.in21k_ft_in1k  fold=1  val_acc=82.8%
[8/10] tf_efficientnetv2_s.in21k_ft_in1k  fold=2  val_acc=82.2%
[9/10] tf_efficientnetv2_s.in21k_ft_in1k  fold=3  val_acc=81.6%
[10/10] tf_efficientnetv2_s.in21k_ft_in1k  fold=4  val_acc=82.6%
Predictions: 4914 samples, 126 unique classes


In [9]:
# Build test DataFrame
test_rows = [
    {'id': int(sid.replace('SAMPLE_', '')), 'Pred': int(p)}
    for sid, p in zip(sample_ids, test_preds)
]
test_df = pd.DataFrame(test_rows)

# Load val submission
assert val_csv_candidates, 'No submission.csv found in inputs!'
val_df = pd.read_csv(val_csv_candidates[0])
print(f'Val submission: {val_csv_candidates[0]}  ({len(val_df)} rows)')

# Combine, deduplicate, sort
final_df = (
    pd.concat([val_df, test_df], ignore_index=True)
    .drop_duplicates(subset='id')
    .sort_values('id')
    .reset_index(drop=True)
)

final_df.to_csv(OUT / 'submission_final.csv', index=False)

print(f'\nVal rows:      {len(val_df)}')
print(f'Test rows:     {len(test_df)}')
print(f'Combined rows: {len(final_df)}')
print(f'Pred range:    [{final_df.Pred.min()}, {final_df.Pred.max()}]')
print(f'Unique classes:{final_df.Pred.nunique()} / {CFG.num_classes}')
print(f'\nSaved -> /kaggle/working/submission_final.csv')
print(final_df.head(5))


Val submission: /kaggle/input/models/shakhoyatshujon/saved-checkpoints/pytorch/default/1/submission.csv  (4914 rows)

Val rows:      4914
Test rows:     4914
Combined rows: 9828
Pred range:    [0, 125]
Unique classes:126 / 126

Saved -> /kaggle/working/submission_final.csv
   id  Pred
0   0   121
1   1    72
2   4     9
3   6   101
4   7    81
